In [ ]:
"""
RicePheno-3V4S Dataset Validation and Metadata Preparation 
"""

import pandas as pd
import numpy as np
import yaml
import hashlib
import warnings
import re

from pathlib import Path
from datetime import datetime
from typing import Dict, List

from PIL import Image
import exifread

warnings.filterwarnings("ignore")


class RicePhenoDatasetManager:

    def __init__(self, dataset_root: str):

        self.dataset_root = Path(dataset_root)

        self.metadata_dataframe = None

        # ---------------------------------------------------------
        # Supported image extensions
        # ---------------------------------------------------------
        self.supported_image_extensions = {
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".tif",
            ".tiff"
        }

        # ---------------------------------------------------------
        # Growth stages
        # ---------------------------------------------------------
        self.growth_stage_mapping = {
            "activetillering": "Active_Tillering",
            "panicleinitiation": "Panicle_Initiation",
            "flowering": "Flowering",
            "harvesting": "Harvesting"
        }

        # ---------------------------------------------------------
        # Rice varieties
        # ---------------------------------------------------------
        self.variety_mapping = {
            "co51": "CO-51",
            "cr1009": "CR1009",
            "iwponni": "IW-Ponni"
        }

        # ---------------------------------------------------------
        # Sensor types
        # ---------------------------------------------------------
        self.sensor_mapping = {
            "uav": "uav",
            "dslr": "dslr",
            "mobile": "mobile"
        }

    # =============================================================
    # Utility Functions
    # =============================================================

    def normalize_text(self, text: str) -> str:

        return (
            text.lower()
            .replace("-", "")
            .replace("_", "")
            .replace(" ", "")
        )

    def generate_unique_image_id(self, image_path: Path) -> str:

        relative_path = str(image_path.relative_to(self.dataset_root))

        md5_hash = hashlib.md5(
            relative_path.encode()
        ).hexdigest()[:8]

        return f"IMG_{image_path.stem}_{md5_hash}"

    def calculate_file_hash(self, file_path: Path) -> str:

        md5_hash = hashlib.md5()

        with open(file_path, "rb") as file:

            for chunk in iter(lambda: file.read(4096), b""):
                md5_hash.update(chunk)

        return md5_hash.hexdigest()

    # =============================================================
    # Date Extraction
    # =============================================================

    def extract_date_from_path(self, image_path: Path) -> str:

        date_regex = re.compile(
            r"(\d{4})[-_](\d{2})[-_](\d{2})"
        )

        for path_part in image_path.parts:

            match = date_regex.search(path_part)

            if match:

                return (
                    f"{match.group(1)}-"
                    f"{match.group(2)}-"
                    f"{match.group(3)}"
                )

        return datetime.fromtimestamp(
            image_path.stat().st_mtime
        ).strftime("%Y-%m-%d")

    # =============================================================
    # EXIF Extraction
    # =============================================================

    def extract_exif_metadata(self, image_path: Path) -> Dict:

        exif_metadata = {}

        try:

            with open(image_path, "rb") as image_file:

                exif_tags = exifread.process_file(
                    image_file,
                    details=False
                )

            # -----------------------------------------------------
            # Camera model
            # -----------------------------------------------------
            if "Image Model" in exif_tags:

                exif_metadata["camera_model"] = str(
                    exif_tags["Image Model"]
                )

            # -----------------------------------------------------
            # ISO
            # -----------------------------------------------------
            if "EXIF ISOSpeedRatings" in exif_tags:

                iso_value = exif_tags["EXIF ISOSpeedRatings"]

                try:
                    exif_metadata["iso"] = int(
                        iso_value.values[0]
                    )
                except:
                    exif_metadata["iso"] = str(iso_value)

            # -----------------------------------------------------
            # Focal Length
            # -----------------------------------------------------
            if "EXIF FocalLength" in exif_tags:

                focal_length = exif_tags["EXIF FocalLength"]

                try:

                    focal_value = (
                        float(focal_length.values[0].num)
                        / float(focal_length.values[0].den)
                    )

                    exif_metadata["focal_length_mm"] = round(
                        focal_value,
                        2
                    )

                except:
                    exif_metadata["focal_length_mm"] = str(
                        focal_length
                    )

            # -----------------------------------------------------
            # Aperture (FIXED)
            # -----------------------------------------------------
            if "EXIF FNumber" in exif_tags:

                aperture_tag = exif_tags["EXIF FNumber"]

                try:

                    # IMPORTANT FIX:
                    # Convert Rational safely
                    aperture_value = (
                        float(aperture_tag.values[0].num)
                        / float(aperture_tag.values[0].den)
                    )

                    # Store as TEXT
                    # Prevent Excel date conversion
                    exif_metadata["aperture_f_number"] = (
                        f"f/{aperture_value:.1f}"
                    )

                except:

                    exif_metadata["aperture_f_number"] = str(
                        aperture_tag
                    )

            # -----------------------------------------------------
            # Exposure Time
            # -----------------------------------------------------
            if "EXIF ExposureTime" in exif_tags:

                exposure_tag = exif_tags["EXIF ExposureTime"]

                try:

                    numerator = exposure_tag.values[0].num
                    denominator = exposure_tag.values[0].den

                    exif_metadata["exposure_time"] = (
                        f"{numerator}/{denominator} sec"
                    )

                except:

                    exif_metadata["exposure_time"] = str(
                        exposure_tag
                    )

            # -----------------------------------------------------
            # Capture Date
            # -----------------------------------------------------
            if "EXIF DateTimeOriginal" in exif_tags:

                exif_metadata["capture_datetime"] = str(
                    exif_tags["EXIF DateTimeOriginal"]
                )

        except Exception as exif_error:

            exif_metadata["exif_error"] = str(
                exif_error
            )

        return exif_metadata

    # =============================================================
    # Image Technical Metadata
    # =============================================================

    def extract_image_metadata(
        self,
        image_path: Path
    ) -> Dict:

        image_metadata = {

            "image_width": 0,
            "image_height": 0,
            "image_channels": 0,
            "image_format": "Unknown",
            "bit_depth": 8,
            "file_size_mb": 0
        }

        try:

            image_metadata["file_size_mb"] = round(
                image_path.stat().st_size / (1024 * 1024),
                2
            )

            with Image.open(image_path) as image:

                image_metadata["image_width"] = image.width

                image_metadata["image_height"] = image.height

                image_metadata["image_channels"] = len(
                    image.getbands()
                )

                image_metadata["image_format"] = image.format

                # -------------------------------------------------
                # Bit depth
                # -------------------------------------------------
                if image.mode == "I;16":

                    image_metadata["bit_depth"] = 16

                elif image.mode in ["RGB", "RGBA"]:

                    image_metadata["bit_depth"] = 24

                else:

                    image_metadata["bit_depth"] = 8

            # -----------------------------------------------------
            # Merge EXIF metadata
            # -----------------------------------------------------
            exif_metadata = self.extract_exif_metadata(
                image_path
            )

            image_metadata.update(exif_metadata)

        except Exception as image_error:

            image_metadata["image_error"] = str(
                image_error
            )

        return image_metadata

    # =============================================================
    # Main Metadata Generation
    # =============================================================

    def generate_metadata_files(

        self,

        csv_output_name="metadata.csv",

        excel_output_name="metadata.xlsx",

        max_images=None
    ):

        metadata_entries = []

        processed_count = 0

        # =========================================================
        # Stage directories
        # =========================================================
        for stage_directory in sorted(
            self.dataset_root.iterdir()
        ):

            if not stage_directory.is_dir():
                continue

            stage_key = self.normalize_text(
                stage_directory.name
            )

            if stage_key not in self.growth_stage_mapping:
                continue

            growth_stage_name = (
                self.growth_stage_mapping[stage_key]
            )

            # =====================================================
            # Variety directories
            # =====================================================
            for variety_directory in sorted(
                stage_directory.iterdir()
            ):

                if not variety_directory.is_dir():
                    continue

                variety_key = self.normalize_text(
                    variety_directory.name
                )

                if variety_key not in self.variety_mapping:
                    continue

                variety_name = (
                    self.variety_mapping[variety_key]
                )

                # =================================================
                # Sensor directories
                # =================================================
                for sensor_directory in sorted(
                    variety_directory.iterdir()
                ):

                    if not sensor_directory.is_dir():
                        continue

                    sensor_key = self.normalize_text(
                        sensor_directory.name
                    )

                    if sensor_key not in self.sensor_mapping:
                        continue

                    sensor_name = (
                        self.sensor_mapping[sensor_key]
                    )

                    # =============================================
                    # Image files
                    # =============================================
                    for image_path in sorted(
                        sensor_directory.iterdir()
                    ):

                        if (
                            image_path.suffix.lower()
                            not in self.supported_image_extensions
                        ):
                            continue

                        # -----------------------------------------
                        # Extract technical metadata
                        # -----------------------------------------
                        technical_metadata = (
                            self.extract_image_metadata(
                                image_path
                            )
                        )

                        # -----------------------------------------
                        # Derived values
                        # -----------------------------------------
                        megapixels = round(

                            (
                                technical_metadata["image_width"]
                                *
                                technical_metadata["image_height"]
                            ) / 1e6,

                            2
                        )

                        aspect_ratio = round(

                            technical_metadata["image_width"]
                            /
                            max(
                                technical_metadata["image_height"],
                                1
                            ),

                            2
                        )

                        # -----------------------------------------
                        # Create metadata row
                        # -----------------------------------------
                        metadata_entries.append({

                            "image_id":
                                self.generate_unique_image_id(
                                    image_path
                                ),

                            "filename":
                                str(
                                    image_path.relative_to(
                                        self.dataset_root
                                    )
                                ),

                            "file_hash":
                                self.calculate_file_hash(
                                    image_path
                                ),

                            "collection_date":
                                self.extract_date_from_path(
                                    image_path
                                ),

                            "variety":
                                variety_name,

                            "growth_stage":
                                growth_stage_name,

                            "sensor_type":
                                sensor_name,

                            "plot_id":
                                f"{variety_name}_{growth_stage_name}",

                            "image_width":
                                technical_metadata.get(
                                    "image_width"
                                ),

                            "image_height":
                                technical_metadata.get(
                                    "image_height"
                                ),

                            "image_channels":
                                technical_metadata.get(
                                    "image_channels"
                                ),

                            "megapixels":
                                megapixels,

                            "aspect_ratio":
                                aspect_ratio,

                            "file_size_mb":
                                technical_metadata.get(
                                    "file_size_mb"
                                ),

                            "image_format":
                                technical_metadata.get(
                                    "image_format"
                                ),

                            "bit_depth":
                                technical_metadata.get(
                                    "bit_depth"
                                ),

                            "camera_model":
                                technical_metadata.get(
                                    "camera_model"
                                ),

                            "focal_length_mm":
                                technical_metadata.get(
                                    "focal_length_mm"
                                ),

                            # FIXED FIELD
                            "aperture_f_number":
                                technical_metadata.get(
                                    "aperture_f_number"
                                ),

                            "iso":
                                technical_metadata.get(
                                    "iso"
                                ),

                            "exposure_time":
                                technical_metadata.get(
                                    "exposure_time"
                                ),

                            "capture_datetime":
                                technical_metadata.get(
                                    "capture_datetime"
                                )
                        })

                        processed_count += 1

                        print(
                            f"Processed: {processed_count}",
                            end="\r"
                        )

                        if (
                            max_images is not None
                            and
                            processed_count >= max_images
                        ):
                            break

        # =========================================================
        # Create DataFrame
        # =========================================================
        metadata_dataframe = pd.DataFrame(
            metadata_entries
        )

        self.metadata_dataframe = metadata_dataframe

        # =========================================================
        # IMPORTANT FIX:
        # Force aperture column as string
        # Prevent Excel auto-date conversion
        # =========================================================
        if "aperture_f_number" in metadata_dataframe.columns:

            metadata_dataframe[
                "aperture_f_number"
            ] = metadata_dataframe[
                "aperture_f_number"
            ].astype(str)

        # =========================================================
        # Save CSV
        # =========================================================
        csv_output_path = (
            self.dataset_root / csv_output_name
        )

        metadata_dataframe.to_csv(

            csv_output_path,

            index=False,

            encoding="utf-8-sig"
        )

        # =========================================================
        # Save Excel
        # =========================================================
        excel_output_path = (
            self.dataset_root / excel_output_name
        )

        with pd.ExcelWriter(
            excel_output_path,
            engine="openpyxl"
        ) as excel_writer:

            metadata_dataframe.to_excel(

                excel_writer,

                index=False,

                sheet_name="RicePheno_Metadata"
            )

            worksheet = excel_writer.sheets[
                "RicePheno_Metadata"
            ]

            # -----------------------------------------------------
            # Prevent Excel date auto-conversion
            # -----------------------------------------------------
            for cell in worksheet["S"]:

                cell.number_format = "@"

        # =========================================================
        # Summary
        # =========================================================
        print("\n")
        print("=" * 60)
        print("METADATA GENERATION COMPLETED")
        print("=" * 60)

        print(
            f"Total Images Processed : "
            f"{len(metadata_dataframe)}"
        )

        print(
            f"CSV Saved  : {csv_output_path}"
        )

        print(
            f"Excel Saved: {excel_output_path}"
        )

        return metadata_dataframe


# =============================================================
# Main Function
# =============================================================

def main():

    dataset_root_directory = (
        "../dataset/RicePheno-3V4S-Dataset_Ver.1.0.0/"
    )

    dataset_manager = RicePhenoDatasetManager(
        dataset_root_directory
    )

    dataset_manager.generate_metadata_files(

        csv_output_name="metadata.csv",

        excel_output_name="metadata.xlsx",

        max_images=None
    )


# =============================================================
# Entry Point
# =============================================================

if __name__ == "__main__":

    main()

Processed: 6045

METADATA GENERATION COMPLETED
Total Images Processed : 6045
CSV Saved  : ..\dataset\RicePheno-3V4S-Dataset_Ver.1.0.0\metadata.csv
Excel Saved: ..\dataset\RicePheno-3V4S-Dataset_Ver.1.0.0\metadata.xlsx
